# Charts for the paper

Recreates every chart from the saved results, without running any model.

Files needed in `FOLDER`: `stage2_results.xlsx`, `crashes_for_breakdowns.csv`, `test_crashes_with_predictions.csv`, `shap_values_test.csv`, and `London_Boroughs.gpkg` for the borough maps.

Run the three cells in order. The charts are saved in `FOLDER/charts_paper` as PDF and PNG.

| Cell | Charts |
|---|---|
| 1 | Setup, style and data |
| 2 | 01 to 11, 17 and 18 |
| 3 | 12_01 to 12_13, 13 to 16 |


In [ ]:
# ============ CELL 1: setup, style and data (run this first) ============
import os, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy.stats import rankdata

warnings.filterwarnings("ignore")

# ---- paths ----
FOLDER = r"..."
BOROUGHS_GPKG = os.path.join(FOLDER, "London_Boroughs.gpkg")      # change if the file is elsewhere
OUT = os.path.join(FOLDER, "charts_paper")
os.makedirs(OUT, exist_ok=True)

# ---- options ----
SHOW_TITLES = True            # False for the paper version (titles go in the figure captions)
FORMATS = ["pdf", "png"]      # add "tiff" if the journal asks for it

# ---- one style for every chart ----
SINGLE, DOUBLE = 3.35, 6.9    # journal column widths in inches (8.5 cm and 17.5 cm)
plt.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 8, "axes.titlesize": 8.5, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.labelsize": 8, "xtick.labelsize": 7.5, "ytick.labelsize": 7.5, "legend.fontsize": 7.5, "legend.frameon": False,
    "axes.spines.top": False, "axes.spines.right": False, "axes.linewidth": 0.6, "xtick.major.width": 0.6,
    "ytick.major.width": 0.6, "xtick.major.size": 3, "ytick.major.size": 3, "lines.linewidth": 1.2,
    "savefig.dpi": 300, "figure.dpi": 110, "pdf.fonttype": 42, "ps.fonttype": 42})
BLUE, LIGHT, ORANGE, GREEN, GREY = "#0F4C81", "#8FB3D3", "#D55E00", "#009E73", "#7A7A7A"
RAISES, LOWERS = "#B2182B", "#2166AC"
SEASON_COLOURS = {"Winter": "#2166AC", "Spring": "#009E73", "Summer": "#E69F00", "Autumn": "#B2182B"}
MODEL_COLOURS = {"XGBoost": BLUE, "Elastic-net logistic regression": ORANGE, "LightGBM": GREEN}
PART_COLOURS = {"Fitting": "#EEF2F6", "Validation": "#FBF0DC", "Test": "#E6F2EC"}
MAP_CMAP = "cividis"
CREDIT = "Contains National Statistics and Ordnance Survey data. Crown copyright and database right."
NAME_FIX = {"One-Way St": "One-way street", "Slip Rd": "Slip road", "Rbt": "Roundabout",
            "City Of London": "City of London", "City Of Westminster": "City of Westminster",
            "Richmond-Upon-Thames": "Richmond upon Thames", "Kingston-Upon-Thames": "Kingston upon Thames",
            "Hammersmith & Fulham": "Hammersmith and Fulham", "Kensington & Chelsea": "Kensington and Chelsea",
            "Barking & Dagenham": "Barking and Dagenham", "missing": "Not recorded"}

def fix(label):
    return NAME_FIX.get(str(label), str(label))

def save(fig, name):
    for ext in FORMATS:
        fig.savefig(os.path.join(OUT, f"{name}.{ext}"), bbox_inches="tight", pad_inches=0.04, dpi=600 if ext == "tiff" else 300)
    plt.show()
    plt.close(fig)
    print(f"Saved {name}")

def title(ax_or_fig, text, **kw):
    if not SHOW_TITLES:
        return
    if hasattr(ax_or_fig, "suptitle"):
        ax_or_fig.suptitle(text, x=0.01, ha="left", fontsize=9, fontweight="bold", **kw)
    else:
        ax_or_fig.set_title(text, **kw)

def panel(ax, letter):
    ax.text(-0.02, 1.02, f"({letter})", transform=ax.transAxes, fontsize=8.5, fontweight="bold", ha="right", va="bottom")

def wilson(k, n, z=1.96):
    if n == 0:
        return np.nan, np.nan
    p, d = k / n, 1 + z * z / n
    c, h = (p + z * z / (2 * n)) / d, z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (c - h) * 100, (c + h) * 100

def wrap(text, width=30):
    return "\n".join(re.findall(rf".{{1,{width}}}(?:\s|$)", str(text))).strip()

# ---- load the saved results ----
EXCEL = os.path.join(FOLDER, "stage2_results.xlsx")
for f in [EXCEL, os.path.join(FOLDER, "crashes_for_breakdowns.csv"), os.path.join(FOLDER, "test_crashes_with_predictions.csv"),
          os.path.join(FOLDER, "shap_values_test.csv")]:
    assert os.path.exists(f), f"Not found: {f}"
SHEET = pd.read_excel(EXCEL, sheet_name=None)
crashes = pd.read_csv(os.path.join(FOLDER, "crashes_for_breakdowns.csv"), low_memory=False, parse_dates=["accident_dt"])
test = pd.read_csv(os.path.join(FOLDER, "test_crashes_with_predictions.csv"), low_memory=False, parse_dates=["accident_dt"])
shap_values = pd.read_csv(os.path.join(FOLDER, "shap_values_test.csv"), low_memory=False)
split = SHEET["Data and split"].set_index("Part")
FINAL = str(SHEET["Settings"].set_index("Item").at["Final model", "Value"])
OVERALL = crashes["serious_or_fatal"].mean() * 100
PERIOD = f"{crashes['accident_dt'].min():%b %Y} to {crashes['accident_dt'].max():%b %Y}"
print(f"Loaded {len(SHEET)} Excel sheets, {len(crashes):,} crashes, {len(test):,} test crashes. Final model: {FINAL}.")
print(f"Charts will be saved in {OUT}")


In [ ]:
# ============ CELL 2: data, model and results charts ============

# ---- 01. Crashes per month by season, the serious share, and the time split ----
months = SHEET["Chart data months"].copy()
months["month"] = pd.to_datetime(months["month"])
months["rolling"] = months["serious %"].rolling(12, center=True, min_periods=6).mean()
fig, axes = plt.subplots(2, 1, figsize=(DOUBLE, 4.0), sharex=True, gridspec_kw={"height_ratios": [1.25, 1], "hspace": 0.12})
for part, colour in PART_COLOURS.items():
    start, end = pd.to_datetime(split.at[part, "From"]), pd.to_datetime(split.at[part, "To"])
    for ax in axes:
        ax.axvspan(start, end, color=colour, linewidth=0, zorder=0)
    axes[0].text(start + (end - start) / 2, 0.97, f"{part}\n{split.at[part, 'Share']}", transform=axes[0].get_xaxis_transform(),
                 ha="center", va="top", fontsize=6.5, color="#444444")
axes[0].bar(months["month"], months["crashes"], width=24, color=[SEASON_COLOURS[s] for s in months["season"]], zorder=2)
axes[0].set_ylabel("Crashes per month")
axes[0].set_ylim(0, months["crashes"].max() * 1.32)
axes[0].legend([Patch(color=c) for c in SEASON_COLOURS.values()], SEASON_COLOURS.keys(), ncol=4, loc="lower left",
               bbox_to_anchor=(0, 1.0), handlelength=1, columnspacing=1)
axes[1].plot(months["month"], months["serious %"], color=LIGHT, linewidth=0.8, label="Month", zorder=2)
axes[1].plot(months["month"], months["rolling"], color=BLUE, linewidth=1.6, label="12-month average", zorder=3)
axes[1].set_ylabel("Serious or fatal (%)\nof the month's crashes")
axes[1].legend(loc="upper left", ncol=2)
panel(axes[0], "a")
panel(axes[1], "b")
title(fig, "Cyclist crashes per month, serious or fatal share and the time split", y=1.04)
save(fig, "01_crashes_by_month_and_season")

# ---- 02. Association with severity and the feature-selection options ----
assoc = SHEET["Feature association"]
options = SHEET["Selection options"]
n_fit = int(split.at["Fitting", "Crashes"])
fig, axes = plt.subplots(1, 2, figsize=(DOUBLE, 2.6), gridspec_kw={"width_ratios": [1, 1.1], "wspace": 0.55})
axes[0].hist(assoc["|correlation|"], bins=40, color=LIGHT, edgecolor="white", linewidth=0.3)
for z, lab, ls in [(1.960, "p < 0.05", ":"), (2.576, "p < 0.01", "--"), (3.291, "p < 0.001", "-.")]:
    axes[0].axvline(z / np.sqrt(n_fit), color=BLUE, linestyle=ls, linewidth=0.8, label=f"{lab} (|r| = {z / np.sqrt(n_fit):.4f})")
axes[0].axvline(0.03, color=ORANGE, linestyle="--", linewidth=0.8, label="|r| = 0.03")
axes[0].legend(loc="upper center", bbox_to_anchor=(0.5, -0.28), fontsize=6.3, ncol=2, title="Thresholds", title_fontsize=6.3)
axes[0].set_xlabel("|Correlation| with serious or fatal injury")
axes[0].set_ylabel("Street features")
opt = options.iloc[::-1]
axes[1].barh(opt["Selection option"], opt["Street features kept"], color=BLUE, height=0.65)
for y_, v in enumerate(opt["Street features kept"]):
    axes[1].text(v + 2, y_, str(int(v)), va="center", fontsize=6.5)
axes[1].set_xlabel("Street features kept")
panel(axes[0], "a")
panel(axes[1], "b")
title(fig, "Association of the street features with severity, and the selection options", y=1.05)
save(fig, "02_feature_association_and_options")

# ---- 03. The model search ----
search = SHEET["Search"]
types = [t for t in MODEL_COLOURS if t in set(search["Model type"])]
fig, axes = plt.subplots(1, 2, figsize=(DOUBLE, 2.9), gridspec_kw={"width_ratios": [1, 1.35], "wspace": 0.6})
rng = np.random.RandomState(1)
for k, t in enumerate(types):
    sc = search.loc[search["Model type"] == t, "Mean ROC-AUC (5 folds)"].values
    axes[0].scatter(k + rng.uniform(-0.22, 0.22, len(sc)), sc, s=6, alpha=0.55, color=MODEL_COLOURS[t], linewidths=0)
    axes[0].scatter(k, sc.max(), s=36, marker="D", color=MODEL_COLOURS[t], edgecolor="black", linewidth=0.5, zorder=3)
axes[0].set_xticks(range(len(types)))
axes[0].set_xticklabels([wrap(t, 14) for t in types])
axes[0].set_ylabel("ROC-AUC, mean of 5 time-ordered folds")
best = search.pivot_table(index="Selection", columns="Model type", values="Mean ROC-AUC (5 folds)", aggfunc="max")
best = best.reindex([o for o in options["Selection option"] if o in best.index])
yy = np.arange(len(best))[::-1]
for k, t in enumerate(types):
    axes[1].scatter(best[t], yy + (k - 1) * 0.2, s=14, color=MODEL_COLOURS[t], label=t)
axes[1].set_yticks(yy)
axes[1].set_yticklabels([wrap(o, 22) for o in best.index], fontsize=7)
axes[1].xaxis.set_major_locator(plt.MaxNLocator(4))
axes[1].xaxis.set_major_formatter(plt.FormatStrFormatter("%.3f"))
axes[1].set_xlabel("Best ROC-AUC (mean of 5 folds)")
axes[1].legend(loc="upper center", bbox_to_anchor=(0.4, -0.2), ncol=3, handletextpad=0.2, columnspacing=0.8)
panel(axes[0], "a")
panel(axes[1], "b")
title(fig, "Model search: every setting (diamond = best) and the best score of each selection option", y=1.05)
save(fig, "03_search_overview")

# ---- 04. Balanced accuracy of every version ----
res = SHEET["Test results"].set_index("Version")
groups = [
    ("Street model", [v for v in res.index if v.startswith("Street")]),
    ("Full model", [v for v in res.index if v.startswith("Full")]),
    ("References", [v for v in res.index if v in ("Safety score only", "Cyclist only")]),
]
rows, colours = [], []
for g, vs in groups:
    for v in vs:
        rows.append(v)
        if g == "References":
            colours.append(GREY)
        elif g == "Full model":
            colours.append(BLUE)
        else:
            colours.append(LIGHT)
view = res.loc[rows]
fig, ax = plt.subplots(figsize=(DOUBLE, 0.26 * len(view) + 0.9))
yy = np.arange(len(view))[::-1]
ax.hlines(yy, view["BA low"], view["BA high"], color=colours, linewidth=1.6)
ax.scatter(
    view["Balanced accuracy %"], yy,
    color=colours, s=22, zorder=3, edgecolor="white", linewidth=0.5,
)
for y_, v, hi in zip(yy, view["Balanced accuracy %"], view["BA high"]):
    ax.text(hi + 0.15, y_, f"{v:.1f}", va="center", fontsize=6.8)
ax.axvline(50, color=GREY, linestyle=":", linewidth=0.8)
ax.text(50, len(view) - 0.4, " chance (50%)", fontsize=6.5, color=GREY, va="bottom")
ax.set_yticks(yy)
ax.set_yticklabels(view.index)
ax.set_xlabel("Balanced accuracy on the test part (%): dot = result, line = 95% range")
x_low = min(49, view["BA low"].min() - 0.5)
x_high = view["BA high"].max() + 1.2
ax.set_xlim(x_low, x_high)
title(ax, f"Balanced accuracy of every version (final model: {FINAL})")
save(fig, "04_balanced_accuracy_versions")

# ---- 05. ROC curves ----
roc = SHEET["Chart data ROC"]
styles = {"Safety score only": (GREY, ":"), "Cyclist only": (GREY, "--"), "Street": (LIGHT, "-"), "Full": (BLUE, "-"),
          "Full + safety mean": (ORANGE, "-")}
fig, ax = plt.subplots(figsize=(SINGLE, SINGLE))
for v, (colour, ls) in styles.items():
    r = roc[roc["Version"] == v]
    if len(r):
        ax.plot(r["False positive rate"], r["True positive rate"], color=colour, linestyle=ls, linewidth=1.1,
                label=f"{v} ({res.at[v, 'ROC-AUC']:.3f})")
ax.plot([0, 1], [0, 1], color="#BBBBBB", linewidth=0.6)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect("equal")
ax.set_xlabel("False positive rate (slight crashes flagged)")
ax.set_ylabel("True positive rate (serious crashes flagged)")
ax.legend(loc="lower right", fontsize=6.5, title="ROC-AUC", title_fontsize=6.5)
title(ax, "ROC curves on the test part")
save(fig, "05_roc_curves")

# ---- 06. With minus without the safety score ----
cmp_ = SHEET["Test comparisons"].copy()
cmp_["label"] = cmp_["Comparison"].str.replace(r" vs (Street|Full)$", "", regex=True)
fig, ax = plt.subplots(figsize=(DOUBLE, 0.28 * len(cmp_) + 0.9))
yy = np.arange(len(cmp_))[::-1]
ax.axvspan(cmp_["Low"].min() - 0.5, 0, color="#F6F6F6", zorder=0)
ax.hlines(yy, cmp_["Low"], cmp_["High"], color=BLUE, linewidth=1.5)
ax.scatter(cmp_["Difference (points)"], yy, color=BLUE, s=18, zorder=3)
main = cmp_["Main question"].astype(str).str.lower().isin(["true", "1"]).values
ax.scatter(cmp_.loc[main, "Difference (points)"], yy[main], s=70, facecolor="none", edgecolor=ORANGE, linewidth=1.2, zorder=4,
           label="Main question (fixed in advance)")
ax.axvline(0, color="black", linewidth=0.7)
ax.set_yticks(yy)
ax.set_yticklabels(cmp_["label"])
ax.set_xlim(cmp_["Low"].min() - 0.3, cmp_["High"].max() + 0.3)
ax.set_xlabel("Change in balanced accuracy (percentage points)\ndot = result, line = 95% range")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22))
title(ax, "Change in balanced accuracy when the safety score or photo timing is added")
save(fig, "06_safety_score_balanced_accuracy")

# ---- 07. Confusion table of the full model ----
yt, flag = test["serious_or_fatal"].values, test["flagged | Full"].values.astype(bool)
cm = np.array([[((~flag) & (yt == 0)).sum(), (flag & (yt == 0)).sum()], [((~flag) & (yt == 1)).sum(), (flag & (yt == 1)).sum()]])
fig, ax = plt.subplots(figsize=(SINGLE, 2.5))
ax.imshow(cm / cm.sum(axis=1, keepdims=True), cmap="Blues", vmin=0, vmax=1)
for (r_, c_), v in np.ndenumerate(cm):
    share = v / cm[r_].sum()
    ax.text(c_, r_, f"{v:,}\n{share * 100:.1f}%", ha="center", va="center", fontsize=8, color="white" if share > 0.5 else "black")
ax.set_xticks([0, 1])
ax.set_xticklabels(["Predicted slight", "Predicted serious\nor fatal"])
ax.set_yticks([0, 1])
ax.set_yticklabels(["Slight", "Serious or fatal"])
ax.set_ylabel("Actual outcome")
for sp in ax.spines.values():
    sp.set_visible(False)
ax.tick_params(length=0)
title(ax, "Full model on the test part (percentages by row)")
save(fig, "07_confusion_full_model")

# ---- 08. Observed severity by predicted risk ----
dec = SHEET["Risk tenths"]
fig, ax = plt.subplots(figsize=(SINGLE, 2.6))
x = dec["tenth"].astype(int)
ax.bar(x, dec["serious %"], color=BLUE, width=0.7)
ax.errorbar(x, dec["serious %"], yerr=[dec["serious %"] - dec["95% low"], dec["95% high"] - dec["serious %"]],
            fmt="none", ecolor="black", elinewidth=0.6, capsize=1.5)
test_rate = yt.mean() * 100
ax.axhline(test_rate, color=ORANGE, linestyle="--", linewidth=0.8, label=f"All test crashes ({test_rate:.1f}%)")
ax.legend(loc="upper left")
ax.set_ylim(0, dec["95% high"].max() * 1.15)
ax.set_xticks(range(1, 11))
ax.set_xlabel("Predicted risk, tenths (1 = lowest)")
ax.set_ylabel("Serious or fatal (%)")
title(ax, "Observed severity by predicted risk")
save(fig, "08_risk_tenths")

# ---- 09. SHAP: the 15 most important inputs, with direction ----
shap_tab = SHEET["SHAP inputs"].copy()
top = shap_tab.head(15).iloc[::-1]
signed = np.where(top["Direction"] == "raises risk", 1, np.where(top["Direction"] == "lowers risk", -1, 0)) * top["Mean |SHAP|"]
fig, ax = plt.subplots(figsize=(DOUBLE, 3.9))
ax.barh([wrap(fix(n), 60) for n in top["Readable name"]], signed, color=[RAISES if v > 0 else LOWERS for v in signed], height=0.65)
ax.axvline(0, color="black", linewidth=0.7)
lim = np.abs(signed).max() * 1.1
ax.set_xlim(-lim, lim)
ax.set_xlabel("Mean |SHAP value| (log-odds of a serious or fatal injury), signed by direction")
ax.legend([Patch(color=RAISES), Patch(color=LOWERS)], ["Raises the risk", "Lowers the risk"], loc="lower right")
title(ax, f"The 15 most important inputs of the full model ({FINAL})")
save(fig, "09_shap_top15_signed")

# ---- 10. SHAP: one dot per test crash ----
names15 = shap_tab.head(15)["Readable name"].tolist()
fig, ax = plt.subplots(figsize=(DOUBLE, 4.3))
rng = np.random.RandomState(1)
for row, nm in enumerate(names15[::-1]):
    sv, xv = shap_values.get(f"SHAP | {nm}"), shap_values.get(f"value | {nm}")
    if sv is None or xv is None:
        continue
    colour = rankdata(xv) / len(xv)
    sel = rng.choice(len(sv), size=min(len(sv), 3000), replace=False)
    ax.scatter(sv.values[sel], row + rng.uniform(-0.28, 0.28, len(sel)), c=colour[sel], cmap="coolwarm", s=2.2, alpha=0.65, linewidths=0)
ax.set_yticks(range(len(names15)))
ax.set_yticklabels([wrap(fix(n), 60) for n in names15[::-1]])
ax.axvline(0, color="black", linewidth=0.7)
ax.set_xlabel("SHAP value (log-odds of a serious or fatal injury), one dot per test crash")
cb = plt.colorbar(plt.cm.ScalarMappable(cmap="coolwarm"), ax=ax, fraction=0.025, pad=0.01)
cb.set_ticks([0, 1])
cb.set_ticklabels(["Low", "High"])
cb.set_label("Value of the input", fontsize=7)
cb.outline.set_linewidth(0.4)
title(ax, "How each input moves the predicted risk")
save(fig, "10_shap_summary_dots")

# ---- 11. SHAP dependence for the top four number inputs ----
numeric = shap_tab[shap_tab["Kind"] == "number"].head(4)["Readable name"].tolist()
numeric = [nm for nm in numeric if f"SHAP | {nm}" in shap_values.columns]
if numeric:
    fig, axes = plt.subplots(1, len(numeric), figsize=(DOUBLE, 2.2), squeeze=False, sharey=True)
    for k, (ax, nm) in enumerate(zip(axes[0], numeric)):
        ax.scatter(shap_values[f"value | {nm}"], shap_values[f"SHAP | {nm}"], s=2, alpha=0.3, color=BLUE, linewidths=0)
        ax.axhline(0, color="black", linewidth=0.6)
        ax.set_xlabel(wrap(fix(nm), 26), fontsize=7)
        panel(ax, "abcd"[k])
    axes[0][0].set_ylabel("SHAP value")
    title(fig, "How the impact changes along the most important number inputs", y=1.06)
    fig.subplots_adjust(wspace=0.12)
    save(fig, "11_shap_dependence")

# ---- 17. Robustness: two time periods ----
periods = SHEET["Time periods"]
def parse_range(s):
    a, b = [float(x) for x in re.findall(r"-?\d+\.?\d*", str(s))[:2]]
    return a, b
fig, ax = plt.subplots(figsize=(SINGLE * 1.3, 2.4))
x = np.arange(len(periods))
for k, (col, rcol, colour, lab) in enumerate([("Street BA %", "Street BA range", LIGHT, "Street"), ("Full BA %", "Full BA range", BLUE, "Full")]):
    lo_hi = np.array([parse_range(s) for s in periods[rcol]])
    xs = x + (k - 0.5) * 0.15
    ax.vlines(xs, lo_hi[:, 0], lo_hi[:, 1], color=colour, linewidth=1.5)
    ax.scatter(xs, periods[col], color=colour, s=22, zorder=3, label=lab)
ax.set_xticks(x)
ax.set_xticklabels([f"{r['From']} to\n{r['To']}" for _, r in periods.iterrows()])
ax.set_ylabel("Balanced accuracy (%)")
ax.legend(loc="lower right")
title(ax, "Balanced accuracy in two test periods")
save(fig, "17_time_periods")

# ---- 18. The models for the paper ----
pm = SHEET["Models for the paper"].copy()
pm["lo"], pm["hi"] = zip(*[parse_range(s) for s in pm["Test BA 95% range"]])
fig, ax = plt.subplots(figsize=(SINGLE * 1.6, 0.42 * len(pm) + 0.7))
yy = np.arange(len(pm))[::-1]
ax.hlines(yy, pm["lo"], pm["hi"], color=BLUE, linewidth=1.5)
ax.scatter(pm["Test balanced accuracy %"], yy, color=BLUE, s=24, zorder=3, label="Test (95% range)")
ax.scatter(pm["Validation balanced accuracy %"], yy, color="white", edgecolor=ORANGE, s=24, zorder=3, linewidth=1.1, label="Validation")
ax.set_yticks(yy)
ax.set_yticklabels([wrap(m, 34) for m in pm["Model"]])
ax.set_xlabel("Balanced accuracy, full model (%)")
ax.legend(loc="upper center", bbox_to_anchor=(0.45, -0.32), ncol=2)
title(ax, "The final model and the other candidates")
save(fig, "18_models_for_the_paper")


In [ ]:
# ============ CELL 3: severity breakdowns and maps ============
BREAKDOWNS = ["Borough", "Inner or Outer London", "Speed limit", "Road type", "Junction type", "Light conditions", "Time of day",
              "Weekday or weekend", "Season", "Cyclist age group", "Cyclist sex", "Deprivation decile", "Perceived safety fifth"]
ORDERED = {"Cyclist age group": ["under 16", "16-24", "25-34", "35-44", "45-54", "55-64", "65+", "Not recorded"],
           "Season": ["Winter", "Spring", "Summer", "Autumn"], "Speed limit": ["20 mph or less", "30 mph", "40 mph or more", "Not recorded"]}

# ---- 12. Share of crashes and serious or fatal share, by group ----
breakdown_tables = []
for k, col in enumerate(BREAKDOWNS):
    if col not in crashes.columns:
        print(f"{col}: not in the file, skipped")
        continue
    g = crashes[col].astype(object).where(crashes[col].notna(), "missing").astype(str).map(fix)
    tab = crashes.groupby(g)["serious_or_fatal"].agg(["size", "sum"]).rename(columns={"size": "n", "sum": "k"}).reset_index()
    tab = tab.rename(columns={tab.columns[0]: "Group"})
    tab["share"] = tab["n"] / tab["n"].sum() * 100
    tab["pct"] = tab["k"] / tab["n"] * 100
    tab[["lo", "hi"]] = [wilson(a, b) for a, b in zip(tab["k"], tab["n"])]
    if col in ORDERED:
        tab = tab.assign(_o=tab["Group"].map(lambda x: ORDERED[col].index(x) if x in ORDERED[col] else 99)).sort_values("_o").drop(columns="_o")
    elif col in ("Deprivation decile", "Perceived safety fifth"):
        tab = tab.sort_values("Group")
    else:
        tab = tab.sort_values("n", ascending=False)
    breakdown_tables.append(tab.assign(Breakdown=col))
    tab = tab[tab["n"] >= 30].iloc[::-1].reset_index(drop=True)
    height = 0.23 * len(tab) + 1.0
    fig, axes = plt.subplots(1, 2, figsize=(DOUBLE, height), sharey=True, gridspec_kw={"width_ratios": [1, 1.2], "wspace": 0.06})
    yy = np.arange(len(tab))
    axes[0].barh(yy, tab["share"], color=LIGHT, height=0.65)
    for y_, v, c in zip(yy, tab["share"], tab["n"]):
        axes[0].text(v, y_, f" {v:.1f}% (n = {c:,})", va="center", fontsize=6.3)
    axes[0].set_xlim(0, tab["share"].max() * 1.8)
    axes[0].set_yticks(yy)
    axes[0].set_yticklabels(tab["Group"])
    axes[0].set_xlabel("Share of all crashes (%); groups add up to 100%")
    axes[1].axvline(OVERALL, color=ORANGE, linestyle="--", linewidth=0.8)
    axes[1].hlines(yy, tab["lo"], tab["hi"], color=BLUE, linewidth=1.3)
    axes[1].scatter(tab["pct"], yy, color=BLUE, s=14, zorder=3)
    axes[1].set_xlabel(f"Serious or fatal (%) within the group, 95% range\n(dashed line: all crashes, {OVERALL:.1f}%)")
    panel(axes[0], "a")
    panel(axes[1], "b")
    if SHOW_TITLES:
        fig.suptitle(f"{col}: share of crashes and serious or fatal share ({PERIOD})", x=0.01, ha="left", fontsize=9,
                     fontweight="bold", y=1 + 0.05 / height + 0.02)
    slug = re.sub(r"[^a-z]+", "_", col.lower()).strip("_")
    save(fig, f"12_{k + 1:02d}_severity_{slug}")
pd.concat(breakdown_tables).to_csv(os.path.join(OUT, "breakdown_tables.csv"), index=False)

# ---- 13. Serious or fatal share across the six most important features ----
tf = SHEET["Top feature severity"].copy()
feats = list(dict.fromkeys(tf["Feature"]))[:6]
fig, axes = plt.subplots(2, 3, figsize=(DOUBLE, 5.0))
for k, (ax, f) in enumerate(zip(axes.ravel(), feats)):
    t = tf[tf["Feature"] == f].iloc[::-1]
    yy = np.arange(len(t))
    ax.axvline(OVERALL, color=ORANGE, linestyle="--", linewidth=0.7)
    ax.hlines(yy, t["95% low"], t["95% high"], color=BLUE, linewidth=1.1)
    ax.scatter(t["Serious or fatal within the group %"], yy, color=BLUE, s=11, zorder=3)
    ax.set_yticks(yy)
    ax.set_yticklabels([wrap(fix(gp), 24) for gp in t["Group"]], fontsize=6.3)
    ax.set_title(wrap(fix(t["Readable name"].iloc[0]), 32), fontsize=7.3, fontweight="bold")
    ax.set_xlabel("Serious or fatal (%)", fontsize=7)
    panel(ax, "abcdef"[k])
for ax in axes.ravel()[len(feats):]:
    ax.set_visible(False)
title(fig, f"Serious or fatal share across the six most important features (dashed line: all crashes, {OVERALL:.1f}%)", y=1.01)
plt.tight_layout()
save(fig, "13_severity_top_features")

# ---- maps: borough boundaries (optional) ----
try:
    import geopandas as gpd
    boroughs = gpd.read_file(BOROUGHS_GPKG)
    boroughs = boroughs.set_crs(27700) if boroughs.crs is None else boroughs.to_crs(27700)
    text_cols = [c for c in boroughs.columns if c != "geometry" and boroughs[c].dtype == object]
    NAME_COL = next((c for c in boroughs.columns if c.lower() in ("name", "lad22nm", "lad21nm", "lad23nm", "borough", "nm")), text_cols[0])
except Exception as error:
    boroughs = None
    print(f"Borough boundaries not available ({str(error)[:120]}). Hexagon maps are drawn without boundaries; borough maps are skipped.")

def base_map(ax):
    if boroughs is not None:
        boroughs.boundary.plot(ax=ax, color="#555555", linewidth=0.3)
    ax.set_aspect("equal")
    ax.set_axis_off()

def colourbar(fig, mappable, ax, label):
    cb = fig.colorbar(mappable, ax=ax, fraction=0.035, pad=0.01)
    cb.set_label(label, fontsize=7)
    cb.ax.tick_params(labelsize=6.5)
    cb.outline.set_linewidth(0.4)

xy = crashes.dropna(subset=["easting", "northing"])

# ---- 14. Hexagon maps ----
fig, axes = plt.subplots(1, 2, figsize=(DOUBLE, 2.8))
hb = axes[0].hexbin(xy["easting"], xy["northing"], gridsize=45, mincnt=1, bins="log", cmap=MAP_CMAP, linewidths=0)
base_map(axes[0])
colourbar(fig, hb, axes[0], "Crashes (log scale)")
axes[0].set_title("Crashes", fontsize=8)
hb2 = axes[1].hexbin(xy["easting"], xy["northing"], C=xy["serious_or_fatal"] * 100, reduce_C_function=np.mean, gridsize=30,
                     mincnt=40, cmap=MAP_CMAP, linewidths=0)
base_map(axes[1])
colourbar(fig, hb2, axes[1], "Serious or fatal (%)")
axes[1].set_title("Serious or fatal share (hexagons with 40+ crashes)", fontsize=8)
panel(axes[0], "a")
panel(axes[1], "b")
fig.text(0.01, -0.01, CREDIT, fontsize=5.5, color=GREY)
title(fig, f"Where cyclist crashes happen, and how often they are serious ({PERIOD})", y=1.02)
fig.subplots_adjust(left=0.02, right=0.96, top=0.9, bottom=0.04, wspace=0.12)
save(fig, "14_maps_hexagons")

# ---- 15. Borough maps ----
if boroughs is not None:
    pts = gpd.GeoDataFrame(xy[["serious_or_fatal", "safety_score_mean"]].copy(), geometry=gpd.points_from_xy(xy["easting"], xy["northing"]), crs=27700)
    joined = gpd.sjoin(pts, boroughs[[NAME_COL, "geometry"]], how="left", predicate="within")
    stats = joined.groupby(NAME_COL).agg(n=("serious_or_fatal", "size"), k=("serious_or_fatal", "sum"), safety=("safety_score_mean", "mean")).reset_index()
    stats["share"] = stats["n"] / stats["n"].sum() * 100
    stats["pct"] = stats["k"] / stats["n"] * 100
    tt = test.dropna(subset=["easting", "northing"]).copy()
    tt["risk_pct"] = tt["risk | Full"].rank(pct=True) * 100
    tpts = gpd.GeoDataFrame(tt[["risk_pct"]], geometry=gpd.points_from_xy(tt["easting"], tt["northing"]), crs=27700)
    tj = gpd.sjoin(tpts, boroughs[[NAME_COL, "geometry"]], how="left", predicate="within")
    stats = stats.merge(tj.groupby(NAME_COL)["risk_pct"].mean().reset_index(), on=NAME_COL, how="left")
    mapdf = boroughs.merge(stats, on=NAME_COL, how="left")
    panels = [("share", "%", "Share of all crashes"), ("pct", "%", "Serious or fatal share of the borough's crashes"),
              ("safety", "Score", "Average perceived safety (Paper 1 score)"), ("risk_pct", "Percentile", "Average predicted risk (test crashes)")]
    fig, axes = plt.subplots(2, 2, figsize=(DOUBLE, 5.2))
    for k, (ax, (col, unit, ttl)) in enumerate(zip(axes.ravel(), panels)):
        mapdf.plot(column=col, cmap=MAP_CMAP, ax=ax, edgecolor="white", linewidth=0.3, missing_kwds={"color": "#EEEEEE"})
        sm = plt.cm.ScalarMappable(cmap=MAP_CMAP, norm=plt.Normalize(mapdf[col].min(), mapdf[col].max()))
        colourbar(fig, sm, ax, unit)
        ax.set_axis_off()
        ax.set_title(wrap(ttl, 40), fontsize=7.5)
        panel(ax, "abcd"[k])
    fig.text(0.01, -0.01, CREDIT, fontsize=5.5, color=GREY)
    title(fig, "London boroughs: crashes, severity, perceived safety and predicted risk", y=1.0)
    fig.subplots_adjust(left=0.02, right=0.96, top=0.92, bottom=0.03, wspace=0.12, hspace=0.18)
    save(fig, "15_maps_boroughs")
    stats.rename(columns={NAME_COL: "Borough", "n": "Crashes", "k": "Serious or fatal", "share": "Share of all crashes %",
                          "pct": "Serious or fatal %", "safety": "Average perceived safety", "risk_pct": "Average predicted risk percentile"}
                 ).to_csv(os.path.join(OUT, "borough_map_data.csv"), index=False)

# ---- 16. Test crashes by predicted risk ----
tt = test.dropna(subset=["easting", "northing"]).copy()
tt["risk_pct"] = tt["risk | Full"].rank(pct=True) * 100
tt = tt.sort_values("risk_pct")
fig, ax = plt.subplots(figsize=(DOUBLE * 0.72, 3.6))
sc = ax.scatter(tt["easting"], tt["northing"], c=tt["risk_pct"], cmap="coolwarm", s=1.4, alpha=0.8, linewidths=0)
base_map(ax)
colourbar(fig, sc, ax, "Predicted risk (percentile)")
fig.text(0.01, 0.0, CREDIT, fontsize=5.5, color=GREY)
title(ax, "Test crashes coloured by the full model's predicted risk")
save(fig, "16_map_predicted_risk")
print(f"\nAll charts saved in {OUT}")
